# Deep Learning Assignment 1
Submitters:
1. Amit Ner-Gaon 211649801
2. Chen Frydman 208009845

The purpose of this assignment is to build a simple neural network “from scratch” and to obtain a deep understanding of the forward/backward propagation process.

# Imports and Constants

In [ ]:
import numpy as np

SEED = 42

# Section 1 - Forward Propagation

In [ ]:
def initialize_parameters(layer_dims: list) -> dict:
    """
    Args:
        layer_dims (list): An array of the dimensions of each layer in the network (layer 0 is the size of the flattened input, layer L is the output softmax).
    Returns:
        parameters (dict): a dictionary containing the initialized W and b parameters of each layer {layer_number : {W,b}}.
    """
    rng = np.random.RandomState(SEED)  # for reproducibility

    parameters = {}  # {layer_number : {W,b}}

    L = (
        len(layer_dims) - 1
    )  # number of layers in the network (excluding the input layer)

    for l in range(
        1, L + 1
    ):  # starting from 1 because layer 0 is the input layer, which doesn't have parameters
        n_curr = layer_dims[l]
        n_prev = layer_dims[l - 1]

        current_weights = rng.normal(loc=0.0, scale=0.1, size=(n_curr, n_prev))

        parameters[l] = {
            "W": current_weights,
            "b": np.zeros((n_curr, 1)),  # bias is a vector of size n_curr
        }

    return parameters

In [ ]:
def linear_forward(
    A: np.ndarray, W: np.ndarray, b: np.ndarray
) -> tuple[np.ndarray, dict]:
    """
    Args:
        A (np.ndarray): activations from previous layer (or input data): (size of previous layer, number of examples)
        W (np.ndarray): weights matrix, numpy array of shape (size of current layer, size of previous layer)
        b (np.ndarray): bias vector, numpy array of shape (size of the current layer, 1)
    Returns:
        Z (np.ndarray): the input of the activation function, also called pre-activation parameter
        linear_cache (dict): a python dictionary containing "A", "W" and "b" ; stored for computing the backward pass efficiently
    """
    Z = np.dot(W, A) + b  # Z = WA + b
    linear_cache: dict = {"A": A, "W": W, "b": b}
    return Z, linear_cache

$$Softmax(z)_i = \frac{\exp(z_i)}{\sum_j \exp(z_j)}$$

In [ ]:
def softmax(Z: np.ndarray) -> tuple[np.ndarray, dict]:
    """
    Args:
        Z (np.ndarray): The linear component of the activation function, of shape (number of classes, number of examples)
    Returns:
        A (np.ndarray): The output of the softmax activation function, of shape (number of classes, number of examples)
        activation_cache (dict): a python dictionary containing "Z", stored for computing the backward pass efficiently
    """
    shifted_Z = Z - np.max(
        Z, axis=0, keepdims=True
    )  # we substract maximum Z in order to avoid inf
    exp_Z = np.exp(shifted_Z)
    A = exp_Z / np.sum(
        exp_Z, axis=0, keepdims=True
    )  # axis=0 for sum down, keepdims for keeping the shape
    activation_cache: dict = {"Z": Z}
    return A, activation_cache

$$g(z)=max(0,z)$$
$$g'(z) = \begin{cases} 1 & \textit{if } z > 0 \\ 0 & \textit{if } z < 0 \end{cases}$$

In [ ]:
def relu(Z: np.ndarray) -> tuple[np.ndarray, dict]:
    """
    Args:
        Z (np.ndarray): the linear component of the activation function
    Returns:
        A (np.ndarray): the activations of the layer
        activiation_cache (dict): dict with Z
    """
    A = np.maximum(0, Z)
    activation_cache = {"Z": Z}
    return A, activation_cache

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

In [ ]:
def sigmoid(Z: np.ndarray) -> tuple[np.ndarray, dict]:
    """
    Args:
        Z (np.ndarray): the linear component of the activation function
    Returns:
        A (np.ndarray): the activations of the layer
        activiation_cache (dict): dict with Z
    """
    A = 1 / (1 + np.exp(-Z))
    activation_cache: dict = {"Z": Z}
    return A, activation_cache

In [ ]:
def linear_activation_forward(
    A_prev: np.ndarray, W: np.ndarray, B: np.ndarray, activation: str
) -> tuple[np.ndarray, dict]:
    """
    Implement the forward propagation for the LINEAR->ACTIVATION layer
    Args:
        A_prev (np.ndarray): activations of the previous layer
        W (np.ndarray): the weights matrix of the current layer
        B (np.ndarray): the bias vector of the current layer
        activation (str): the activation function to be used (a string, either “softmax” or “relu”)
    Returns:
        A (np.ndarray):  the activations of the current layer
        cache (dict): a joint dictionary containing both linear_cache and activation_cache
    """
    if activation.lower() == "relu":
        Z, linear_cache = linear_forward(A_prev, W, B)
        A, activation_cache = relu(Z)
    elif activation.lower() == "sigmoid":
        Z, linear_cache = linear_forward(A_prev, W, B)
        A, activation_cache = sigmoid(Z)
    else:
        raise ValueError("Unsupported activation function. Use 'relu' or 'sigmoid'.")
    cache: dict = {"linear_cache": linear_cache, "activation_cache": activation_cache}
    return A, cache

In [ ]:
def l_model_forward(
    X: np.ndarray, parameters: dict, use_batchnorm: bool
) -> tuple[np.ndarray, list]:
    """
    Implement forward propagation for the [LINEAR->RELU]*(L-1)->LINEAR->SOFTMAX computation
    Args:
        X (np.ndarray): data, numpy array of shape (input size, number of examples)
        parameters (dict): the initialized W and b parameters of each layer
        use_batchnorm (bool): whether to use batch normalization or not
    Returns:
        AL (np.ndarray): last post-activation value
        caches (list): a list of all the cache objects generated by the linear_forward function
    """
    caches = []
    A = X
    L = len(parameters)  # number of layers in the neural network

    for l in range(1, L):  # loop from 1 to L-1
        A_prev = A
        W = parameters[l]["W"]
        b = parameters[l]["b"]
        A, cache = linear_activation_forward(A_prev, W, b, activation="relu")
        if use_batchnorm:
            A = apply_batchnorm(A)
        caches.append(cache)

    # Final layer with softmax activation
    AL, cache = linear_activation_forward(A, W, b, activation="softmax")
    caches.append(cache)

    return AL, caches

$$cost = -\frac{1}{m}\sum_{m=1}^{M}\sum_{c=1}^{C}y_i\log(\hat{y})$$

In [ ]:
def compute_cost(AL: np.ndarray, Y: np.ndarray):
    """
    Args:
        AL (np.ndarray): probability vector corresponding to the label predictions, shape (number of classes, number of examples)
        Y (np.ndarray): the labels vector (i.e. the ground truth), shape (number of classes, number of examples)
    Returns:
        cost (float): the cross-entropy cost
    """
    m = Y.shape[1]  # number of examples
    cost: float = -np.sum(Y * np.log(AL + 1e-8)) / m  # adding epsilon to avoid log(0)
    return cost

In [ ]:
def apply_batchnorm(A: np.ndarray) -> np.ndarray:
    """
    performs batchnorm on the received activation values of a given layer.
    Args:
        A (np.ndarray): the activations of a given layer, of shape (size of the layer, number of examples)
    Returns:
        NA (np.ndarray):  the normalized activation values, based on the formula learned in class
    """
    # axis = 1 for row-wise calculation, meaning for each feature across all the examples
    mu = np.mean(A, axis=1, keepdims=True)
    var = np.var(A, axis=1, keepdims=True)

    NA = (A - mu) / np.sqrt(var + 1e-8)

    return NA

# Section 2 - Backward Propagation

In [ ]:
def linear_backward(dZ, cache):
    raise NotImplementedError("This function is not implemented yet.")

In [ ]:
def linear_activation_backward(dA, cache, activation):
    raise NotImplementedError("This function is not implemented yet.")

In [ ]:
def relu_backward(dA, activation_cache):
    raise NotImplementedError("This function is not implemented yet.")

In [ ]:
def softmax_backward(dA, activation_cache):
    raise NotImplementedError("This function is not implemented yet.")

In [ ]:
def l_model_backward(AL, Y, caches):
    raise NotImplementedError("This function is not implemented yet.")

In [ ]:
def update_parameters(parameters, grads, learning_rate):
    raise NotImplementedError("This function is not implemented yet.")

# Section 3 - Full Training Loop

In [ ]:
def l_layer_model(X, Y, layers_dims, learning_rate, num_iterations, batch_size):
    raise NotImplementedError("This function is not implemented yet.")

In [ ]:
def predict(X, Y, parameters):
    raise NotImplementedError("This function is not implemented yet.")

# Section 4 - Experiments

## Section 4.a

## Section 4.b

## Section 4.c

## Section 4.d

# Section 5 - Conclusions